In [1]:
import pandas as pd

In [2]:
import pickle

In [3]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error

In [10]:
import mlflow
mlflow.set_tracking_uri('http://localhost:5001')
mlflow.set_experiment('my-new-experiment-2')

2026/05/28 10:11:26 INFO mlflow.tracking.fluent: Experiment with name 'my-new-experiment-2' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/julia/Desktop/Projects/mlops-zoomcamp-exercises/03-ml-pipelines/artifacts/2', creation_time=1779930686257, experiment_id='2', last_update_time=1779930686257, lifecycle_stage='active', name='my-new-experiment-2', tags={}, trace_location=None, workspace='default'>

In [14]:
def read_dataframe(filename): 
    df = pd.read_parquet(filename)
    
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    
    df = df[((df.duration >= 0) & (df.duration <= 60))]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']

    return df

In [18]:
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet')

In [ ]:
categorical = ['PU_DO']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.fit_transform(val_dicts)

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [20]:
import xgboost as xgb

In [21]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [22]:
mlflow.xgboost.autolog(disable=True)

In [23]:
with mlflow.start_run():
    best_params = {
        'learning_rate': 0.23663837413933345,
        'max_depth': 63,
        'min_child_weight': 5.145379319889727,
        'objective': 'reg:linear',
        'reg_alpha': 0.011421312210525376,
        'reg_lambda': 0.365514542783101,
        'seed': 42
    }
    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=30,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )  
    y_pred = booster.predict(valid) 
    rmse = root_mean_squared_error(y_val, y_pred)
    
    mlflow.xgboost.log_model(booster, artifact_path='models_mlflow')
    # mlflow.log_artifact(local_path='../models/lin_reg.bin', artifact_path='models_pickle')

/Users/julia/miniforge3/envs/mlops_bootcamp/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [10:36:27] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:13.07861
[1]	validation-rmse:15.03831
[2]	validation-rmse:17.08390
[3]	validation-rmse:18.82563
[4]	validation-rmse:20.24474
[5]	validation-rmse:21.37040
[6]	validation-rmse:22.25428
[7]	validation-rmse:22.93640
[8]	validation-rmse:23.45478
[9]	validation-rmse:23.85372
[10]	validation-rmse:24.15961
[11]	validation-rmse:24.39432
[12]	validation-rmse:24.57433
[13]	validation-rmse:24.71062
[14]	validation-rmse:24.80872
[15]	validation-rmse:24.94876
[16]	validation-rmse:25.09433
[17]	validation-rmse:25.23666
[18]	validation-rmse:25.26733
[19]	validation-rmse:25.29852
[20]	validation-rmse:25.45870
[21]	validation-rmse:25.62142
[22]	validation-rmse:25.76804
[23]	validation-rmse:25.76873
[24]	validation-rmse:25.77097
[25]	validation-rmse:25.77062
[26]	validation-rmse:25.77617
[27]	validation-rmse:25.77598
[28]	validation-rmse:25.77625
[29]	validation-rmse:25.77781


2026/05/28 10:36:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run nosy-fawn-827 at: http://localhost:5001/#/experiments/2/runs/3e3bd980758541a3a0ce8288480ffc77
🧪 View experiment at: http://localhost:5001/#/experiments/2
